In [1]:
#Test the libraries

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

import xgboost as xgb
import shap

print("All libraries imported successfully!")

Matplotlib is building the font cache; this may take a moment.


All libraries imported successfully!


In [2]:
# Load the modeling table

df = pd.read_parquet(
    'data/processed/modeling_table.parquet'
)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 100000
Columns: 24


In [3]:
#Check the data

df.head()

,customer_id,first_purchase,last_purchase,total_orders,total_spend,avg_order_value,recent_90d_orders,prior_90d_orders,preferred_channel,tenure_days,...,Active,club_member_status,fashion_news_frequency,age,postal_code,usage_change_pct,customer_satisfaction_score,complaints_last_60_days,support_tickets,payment_failures
0,6b76f530c64a43f194f32051291e573cb274c9a7cd0c2a...,2018-09-26,2019-05-28,9,0.132729,0.014748,0,0,1,607,...,1.0,ACTIVE,Regularly,33.0,75c4921a65f14b3ca2f7b741258171c84fedb5b87566bd...,0.0,6.0,0,0,0
1,d5d78cc39dba1e3096a39337487e6953f3a88ccdba5a18...,2018-12-14,2018-12-14,1,0.091508,0.091508,0,0,1,528,...,0.0,ACTIVE,Regularly,28.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,0.0,5.4,1,1,0
2,8cae5ce5bbd2cb6dc7a4315309c88ce1e5c7fcf4818a5b...,2019-10-24,2020-05-17,10,0.333729,0.033373,7,0,2,214,...,0.0,PRE-CREATE,NONE,33.0,47e3ff3b34ab1c2450822a47b85bbdf6316aa5b8487c3e...,200.0,8.1,1,1,0
3,61b32813f52495f84f70537e414517360687899b5cedba...,2019-11-16,2019-11-27,10,0.485424,0.048542,0,0,2,191,...,1.0,ACTIVE,Regularly,57.0,b031a69e442f52fd966ed97584dbe53a3115cfdfafa8a1...,0.0,7.5,1,1,0
4,65b51ae24c0d98a9e9d322fa4b3ae733ee8726c8dee54e...,2019-12-01,2019-12-01,3,0.078339,0.026113,0,3,2,176,...,1.0,ACTIVE,Regularly,62.0,2c29ae653a9282cce4151bd87643c907644e09541abc28...,-100.0,3.8,0,0,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   customer_id                  100000 non-null  str           
 1   first_purchase               100000 non-null  datetime64[us]
 2   last_purchase                100000 non-null  datetime64[us]
 3   total_orders                 100000 non-null  int64         
 4   total_spend                  100000 non-null  float64       
 5   avg_order_value              100000 non-null  float64       
 6   recent_90d_orders            100000 non-null  int64         
 7   prior_90d_orders             100000 non-null  int64         
 8   preferred_channel            100000 non-null  int64         
 9   tenure_days                  100000 non-null  int64         
 10  days_since_last_purchase     100000 non-null  int64         
 11  churn                        100000 no

In [5]:
df['churn'].value_counts()

churn
1    59199
0    40801
Name: count, dtype: int64

In [6]:
df['churn'].value_counts(normalize=True)

churn
1    0.59199
0    0.40801
Name: proportion, dtype: float64

In [7]:
# Define feature columns

numeric_cols = [
    'total_orders',
    'total_spend',
    'avg_order_value',
    'recent_90d_orders',
    'prior_90d_orders',
    'tenure_days',
    'days_since_last_purchase',
    'category_diversity',
    'FN',
    'Active',
    'age',
    'usage_change_pct',
    'customer_satisfaction_score',
    'complaints_last_60_days',
    'support_tickets',
    'payment_failures',
    'preferred_channel'
]

categorical_cols = [
    'club_member_status',
    'fashion_news_frequency'
]

In [8]:
# Check for missing columns

required_cols = numeric_cols + categorical_cols + ['churn', 'customer_id']

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

print("Missing columns:", missing_cols)

Missing columns: []


In [ ]:
# Create the modeling datasets/table converts categorical variables into numeric dummy variables
# This is necessary because machine-learning models need numerical inputs

model_df = pd.get_dummies(
    df[numeric_cols + categorical_cols + ['churn', 'customer_id']],
    columns=categorical_cols,
    drop_first=True
)

print("Modeling table shape:", model_df.shape)

Modeling table shape: (100000, 24)


In [10]:
# Split the data into features (X) and target (y)
# X These are the variables used to predict churn
# y This is the thing we're predicting


X = model_df.drop(
    columns=['churn', 'customer_id']
)

y = model_df['churn']

ids = model_df['customer_id']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (100000, 22)
y shape: (100000,)


In [ ]:
# Split the data into training and testing sets

# we'll divide our customers into: 80% training and 20% testing
# The test set is important because we haven't allowed the models to learn from it
# That gives us a more honest comparison


X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y,
    ids,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training customers:", len(X_train))
print("Testing customers:", len(X_test))

Training customers: 80000
Testing customers: 20000


In [12]:
# Model 1: Logistic Regression
# Scale the numeric features

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [13]:
# Train the logistic regression model

logreg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logreg.fit(X_train_s, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [ ]:
# Get Logistic Regression probabilities for the test set
# Probabilities are the model's confidence that a customer will churn. A probability of 0.8 means the model is 80% confident that the customer will churn.


logreg_probs = logreg.predict_proba(X_test_s)[:, 1]

print(logreg_probs[:10])

[0.78465055 0.93445528 0.90656812 0.66234544 0.75587688 0.9144232
 0.46795122 0.8472971  0.92435399 0.96487408]
